
# Итоговая работа: прогнозирование почасового энергопотребления AEP

Этот notebook полностью воспроизводит итоговую работу:

1. подготовка данных и EDA;
2. ≥5 статистических моделей через `statsforecast`;
3. 3 ML-модели через `mlforecast`;
4. 3 DL-модели через `neuralforecast`;
5. backtesting, метрики, интервальные оценки;
6. итоговое сравнение.

**Рекомендуется запускать в Google Colab.**


In [ ]:

# Установка библиотек (в Colab выполнить один раз)
!pip -q install statsforecast mlforecast neuralforecast lightgbm xgboost


In [ ]:

import os, sys, time, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.stattools import adfuller, acf
from statsmodels.tsa.seasonal import STL

FAST_MODE = True   # False = использовать весь датасет (намного дольше)
H = 24
SEASON = 24
WEEK = 168
N_WINDOWS = 5

DATA_URL = "https://raw.githubusercontent.com/pplonski/datasets-for-start/refs/heads/master/aep-hourly-energy-consumption/AEP_hourly.csv"

os.makedirs("data", exist_ok=True)
os.makedirs("results", exist_ok=True)

def calc_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    err = y_true-y_pred
    return {
        "MAE": np.mean(np.abs(err)),
        "RMSE": np.sqrt(np.mean(err**2)),
        "MAPE": np.mean(np.abs(err)/np.maximum(np.abs(y_true), 1e-8))*100,
        "sMAPE": np.mean(2*np.abs(err)/np.maximum(np.abs(y_true)+np.abs(y_pred),1e-8))*100,
    }



## Задача №1 — загрузка и подготовка данных

Приводим ряд к формату Nixtla: `unique_id`, `ds`, `y`.


In [ ]:

raw = pd.read_csv(DATA_URL)
raw["Datetime"] = pd.to_datetime(raw["Datetime"], errors="coerce")
raw = raw.dropna(subset=["Datetime", "AEP_MW"]).sort_values("Datetime")
raw = raw.drop_duplicates("Datetime", keep="last").set_index("Datetime").asfreq("h")

missing_before = int(raw["AEP_MW"].isna().sum())
raw["AEP_MW"] = raw["AEP_MW"].interpolate(limit=3).ffill().bfill()

if FAST_MODE:
    raw = raw.tail(24*365)   # последний год

df = raw.reset_index().rename(columns={"Datetime":"ds", "AEP_MW":"y"})
df["unique_id"] = "AEP"
df = df[["unique_id","ds","y"]]
df.to_csv("data/aep_prepared.csv", index=False)

print("Наблюдений:", len(df))
print("Период:", df.ds.min(), "—", df.ds.max())
print("Пропусков до заполнения:", missing_before)
display(df.head())


### Базовые описательные статистики

In [ ]:

display(df["y"].describe().to_frame("AEP_MW"))
plt.figure(figsize=(15,4))
plt.plot(df["ds"], df["y"])
plt.title("AEP: почасовое энергопотребление")
plt.xlabel("Время")
plt.ylabel("MW")
plt.show()


### Суточная и недельная сезонность

In [ ]:

tmp = df.copy()
tmp["hour"] = tmp["ds"].dt.hour
tmp["dow"] = tmp["ds"].dt.dayofweek

hour_profile = tmp.groupby("hour")["y"].mean()
dow_profile = tmp.groupby("dow")["y"].mean()

plt.figure(figsize=(10,4))
plt.plot(hour_profile.index, hour_profile.values, marker="o")
plt.title("Средний профиль по часу суток")
plt.xlabel("Час")
plt.ylabel("MW")
plt.grid(alpha=.3)
plt.show()

plt.figure(figsize=(8,4))
plt.bar(dow_profile.index, dow_profile.values)
plt.title("Среднее потребление по дню недели")
plt.xlabel("0=Пн ... 6=Вс")
plt.ylabel("MW")
plt.show()


### ACF, STL и стационарность

In [ ]:

# Для скорости ACF и STL берём ограниченный фрагмент
sample = df["y"].tail(24*60).values
acf_vals = acf(sample, nlags=24*7, fft=True)

plt.figure(figsize=(14,4))
plt.stem(range(len(acf_vals)), acf_vals)
plt.title("ACF до 168 часов")
plt.xlabel("Лаг")
plt.ylabel("ACF")
plt.show()

stl_df = df.set_index("ds")["y"].tail(24*60)
stl = STL(stl_df, period=24, robust=True).fit()
fig = stl.plot()
fig.set_size_inches(14,8)
plt.show()

adf_level = adfuller(df["y"].tail(min(len(df), 24*365)), autolag="AIC")
adf_diff = adfuller(df["y"].diff().dropna().tail(min(len(df)-1, 24*365)), autolag="AIC")

eda_summary = pd.DataFrame([{
    "n_obs": len(df),
    "date_min": df.ds.min(),
    "date_max": df.ds.max(),
    "mean_MW": df.y.mean(),
    "std_MW": df.y.std(),
    "min_MW": df.y.min(),
    "max_MW": df.y.max(),
    "missing_before_fill": missing_before,
    "ADF_level_pvalue": adf_level[1],
    "ADF_diff1_pvalue": adf_diff[1],
}])
eda_summary.to_csv("results/eda_summary.csv", index=False)
display(eda_summary)



### Вывод по задаче №1

Ряд является часовым и демонстрирует выраженную внутрисуточную и недельную структуру. Поэтому в дальнейшем используются `season_length=24`, недельный лаг `168`, а для MSTL — две сезонности `[24, 168]`. Результаты ADF позволяют отдельно оценить стационарность уровня и первой разности.



# Задача №2 — StatsForecast

Сравниваем SeasonalNaive и шесть статистических моделей. Это перекрывает требование ≥5 статистических методов.


In [ ]:

from statsforecast import StatsForecast
from statsforecast.models import (
    SeasonalNaive, ARIMA, AutoARIMA, AutoETS, AutoTheta,
    HoltWinters, MSTL
)

train = df.iloc[:-H].copy()
test = df.iloc[-H:].copy()

models = [
    SeasonalNaive(season_length=24),
    ARIMA(order=(2,1,2), season_length=24),
    AutoARIMA(season_length=24),
    AutoETS(season_length=24),
    AutoTheta(season_length=24),
    HoltWinters(season_length=24),
    MSTL(season_length=[24,168], trend_forecaster=AutoARIMA(season_length=24)),
]

sf = StatsForecast(models=models, freq="h", n_jobs=1)

t0 = time.perf_counter()
stat_fcst = sf.forecast(df=train, h=H, level=[80,95])
stat_runtime = time.perf_counter()-t0

display(stat_fcst.head())
print("Runtime, sec:", round(stat_runtime,2))


In [ ]:

# Определяем колонки точечных прогнозов
id_cols = {"unique_id","ds"}
interval_tokens = ("-lo-80","-hi-80","-lo-95","-hi-95")
point_cols = [
    c for c in stat_fcst.columns
    if c not in id_cols and not any(tok in c for tok in interval_tokens)
]

stat_rows = []
for model in point_cols:
    m = calc_metrics(test["y"], stat_fcst[model])
    m["model"] = model
    m["family"] = "Stat/Baseline"
    stat_rows.append(m)

stat_metrics = pd.DataFrame(stat_rows).sort_values("MAE")
stat_metrics.to_csv("results/statistical_metrics.csv", index=False)
display(stat_metrics)


### Визуализация статистических прогнозов

In [ ]:

plot_df = test[["ds","y"]].merge(stat_fcst, on="ds", how="left")
plt.figure(figsize=(14,6))
plt.plot(plot_df["ds"], plot_df["y"], label="Actual", linewidth=3)
for c in point_cols:
    plt.plot(plot_df["ds"], plot_df[c], label=c, alpha=.8)
plt.title("Статистические модели: прогноз 24 часа")
plt.xticks(rotation=45)
plt.legend(ncol=2)
plt.tight_layout()
plt.show()


### Rolling backtesting

In [ ]:

cv = sf.cross_validation(
    df=df,
    h=H,
    step_size=H,
    n_windows=N_WINDOWS,
    level=[80,95]
)

cv_point_cols = [
    c for c in cv.columns
    if c not in {"unique_id","ds","cutoff","y"}
    and not any(tok in c for tok in interval_tokens)
]

cv_rows = []
for cutoff, grp in cv.groupby("cutoff"):
    for model in cv_point_cols:
        m = calc_metrics(grp["y"], grp[model])
        m.update({"cutoff": cutoff, "model": model})
        cv_rows.append(m)
cv_metrics = pd.DataFrame(cv_rows)
display(cv_metrics.groupby("model")[["MAE","RMSE","MAPE","sMAPE"]].agg(["mean","std"]).round(2))


### Проверка покрытия вероятностных интервалов

In [ ]:

coverage_rows = []
for model in cv_point_cols:
    lo80, hi80 = f"{model}-lo-80", f"{model}-hi-80"
    lo95, hi95 = f"{model}-lo-95", f"{model}-hi-95"
    row = {"model":model}
    if lo80 in cv and hi80 in cv:
        row["coverage_80"] = ((cv.y >= cv[lo80]) & (cv.y <= cv[hi80])).mean()
    if lo95 in cv and hi95 in cv:
        row["coverage_95"] = ((cv.y >= cv[lo95]) & (cv.y <= cv[hi95])).mean()
    coverage_rows.append(row)
coverage = pd.DataFrame(coverage_rows)
display(coverage)



### Вывод по задаче №2

Финальный выбор статистической модели делается по среднему MAE на rolling backtesting, а не только по одному тестовому окну. Дополнительно проверяются разброс MAE между окнами и покрытие 80/95% интервалов.



# Задача №3 — MLForecast

Feature engineering:
- лаги 1, 2, 24, 48, 168;
- rolling mean 24 и 168;
- календарные признаки.


In [ ]:

from mlforecast import MLForecast
from mlforecast.lag_transforms import RollingMean
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor

ml_models = {
    "LinearRegression": LinearRegression(),
    "RandomForest": RandomForestRegressor(
        n_estimators=250 if not FAST_MODE else 100,
        max_depth=14,
        random_state=42,
        n_jobs=-1
    ),
    "LightGBM": LGBMRegressor(
        n_estimators=500 if not FAST_MODE else 200,
        learning_rate=0.04,
        num_leaves=31,
        random_state=42,
        verbosity=-1
    ),
}

mlf = MLForecast(
    models=ml_models,
    freq="h",
    lags=[1,2,24,48,168],
    lag_transforms={
        1: [RollingMean(window_size=24)],
        24: [RollingMean(window_size=168)],
    },
    date_features=["hour","dayofweek","day","month"],
)

t0 = time.perf_counter()
mlf.fit(train)
ml_fcst = mlf.predict(H)
ml_runtime = time.perf_counter()-t0
display(ml_fcst.head())
print("Runtime, sec:", round(ml_runtime,2))


In [ ]:

ml_rows = []
for model in ml_models:
    m = calc_metrics(test["y"], ml_fcst[model])
    m.update({"model":model, "family":"ML"})
    ml_rows.append(m)
ml_metrics = pd.DataFrame(ml_rows).sort_values("MAE")
ml_metrics.to_csv("results/ml_metrics.csv", index=False)
display(ml_metrics)

ml_plot = test[["ds","y"]].merge(ml_fcst, on="ds")
plt.figure(figsize=(14,5))
plt.plot(ml_plot.ds, ml_plot.y, label="Actual", linewidth=3)
for c in ml_models:
    plt.plot(ml_plot.ds, ml_plot[c], label=c)
plt.title("MLForecast: 24 часа")
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()



# NeuralForecast — 3 DL-модели

Чтобы runtime в Colab оставался разумным, в `FAST_MODE` количество обучающих шагов уменьшено. Для финальной версии можно выставить `FAST_MODE=False`.


In [ ]:

from neuralforecast import NeuralForecast
from neuralforecast.models import MLP, LSTM, NHITS

steps = 50 if FAST_MODE else 200
input_size = 168

dl_models = [
    MLP(h=H, input_size=input_size, max_steps=steps, random_seed=42),
    LSTM(h=H, input_size=input_size, max_steps=steps, random_seed=42),
    NHITS(h=H, input_size=input_size, max_steps=steps, random_seed=42),
]

nf = NeuralForecast(models=dl_models, freq="h")

t0 = time.perf_counter()
nf.fit(df=train)
dl_fcst = nf.predict()
dl_runtime = time.perf_counter()-t0
display(dl_fcst.head())
print("Runtime, sec:", round(dl_runtime,2))


In [ ]:

dl_names = [c for c in dl_fcst.columns if c not in ["unique_id","ds"]]
dl_rows = []
for model in dl_names:
    m = calc_metrics(test["y"], dl_fcst[model])
    m.update({"model":model, "family":"DL"})
    dl_rows.append(m)
dl_metrics = pd.DataFrame(dl_rows).sort_values("MAE")
dl_metrics.to_csv("results/dl_metrics.csv", index=False)
display(dl_metrics)

dl_plot = test[["ds","y"]].merge(dl_fcst, on="ds")
plt.figure(figsize=(14,5))
plt.plot(dl_plot.ds, dl_plot.y, label="Actual", linewidth=3)
for c in dl_names:
    plt.plot(dl_plot.ds, dl_plot[c], label=c)
plt.title("NeuralForecast: 24 часа")
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()



### Вывод по задаче №3

ML и DL сравниваются на том же test horizon и с теми же метриками, что делает результаты сопоставимыми. Для ML используются осмысленные lag/date features, а DL получает 168 часов истории.


# Задача №4 — единое сравнение и выбор pipeline

In [ ]:

all_metrics = pd.concat([stat_metrics, ml_metrics, dl_metrics], ignore_index=True)
all_metrics = all_metrics.sort_values("MAE").reset_index(drop=True)
all_metrics.to_csv("results/all_models_metrics.csv", index=False)

display(all_metrics.style.format({
    "MAE":"{:.2f}",
    "RMSE":"{:.2f}",
    "MAPE":"{:.2f}",
    "sMAPE":"{:.2f}",
}))

winner = all_metrics.iloc[0]
print(
    f"Лучшая модель на финальном hold-out: {winner['model']} | "
    f"MAE={winner['MAE']:.2f} MW | RMSE={winner['RMSE']:.2f} MW | "
    f"sMAPE={winner['sMAPE']:.2f}%"
)



## Тест производительности

Измеряется wall-clock время основных блоков. Для корректного сравнения DL стоит запускать на одинаковом типе hardware (CPU/GPU).


In [ ]:

performance = pd.DataFrame([
    {"block":"StatsForecast fit+forecast", "seconds":stat_runtime},
    {"block":"MLForecast fit+forecast", "seconds":ml_runtime},
    {"block":"NeuralForecast fit+forecast", "seconds":dl_runtime},
])
display(performance)



## Финальное заключение

После выполнения всех ячеек используйте значения из `all_models_metrics`, `cv_metrics` и `coverage` в устной защите.

Работа демонстрирует полный цикл анализа временного ряда: очистка → EDA → статистические модели → ML → DL → backtesting → оценка неопределённости → выбор итоговой модели.

**Перед сдачей:** выполните `Runtime → Run all`, сохраните notebook с outputs и загрузите обновлённый notebook в GitHub.
